# Étape 2 — HEP au niveau du groupe : activation haute vs basse

Suite de [`laura_hep.ipynb`](laura_hep.ipynb) (méthode, contrôles et pièges détaillés sur P001).

**Question** : sur l'ensemble des participants, la réponse du cerveau au battement cardiaque (HEP)
change-t-elle entre les vidéos à activation haute et basse ? Amplitude, et surtout latence.

**Données** : participants traités par [`hep_group.py`](hep_group.py) (même sélection que Maroua :
25 participants tirés avec la graine 42, + P001 ; jours 1 et 2). Un fichier par participant dans
`HABS_DATA/derivatives/hep/`.

**Logique statistique** : chaque participant donne **une** différence haute − basse ; le test porte
sur ces différences (les participants sont les unités indépendantes, pas les battements).

In [ ]:
import os
os.environ.setdefault("HABS_DATA", r"C:\Users\ehren\habs-data")   # à adapter sur ta machine

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne
from scipy import stats
from hep_utils import HEP_WIN, LAT_WIN, mean_amp, frac_area_latency, peak_latency, _montage

OUT = Path(os.environ["HABS_DATA"]) / "derivatives" / "hep"
plt.rcParams["figure.dpi"] = 90

## 1. Participants inclus

In [ ]:
resume = pd.read_csv(OUT / "resume.csv").sort_values("subj")
files = sorted(OUT.glob("P*.npz"))
D = {f.stem: dict(np.load(f)) for f in files}
subjects = list(D)
first = D[subjects[0]]
times, ch_names = first["times"], [str(c) for c in first["ch_names"]]
cl_idx = [ch_names.index(str(c)) for c in first["cluster"]]

cols = ["subj", "days", "n_videos_notees", "seuil_basse", "seuil_haute", "videos_basse", "videos_haute",
        "battements_basse", "battements_haute", "hr_mediane_bpm", "exclu"]
print(f"{len(subjects)} participants inclus sur {len(resume)} traités")
resume[[c for c in cols if c in resume]]

## 2. Y a-t-il un HEP ? (vrais battements vs faux battements)

Avant de comparer les conditions, on vérifie que la réponse au battement existe dans le groupe :
moyenne sur toutes les vidéos, comparée aux faux battements tirés au hasard.

In [ ]:
def cl(x):                       # (électrodes, temps) -> groupe fronto-central (temps)
    return x[cl_idx].mean(0)

def all_beats(d):               # HEP toutes conditions, pondéré par le nombre de battements
    n = {c: (d["wave_cond"] == c).sum() for c in ["basse", "moyenne", "haute"]}
    return sum(d[f"evoked_{c}"] * n[c] for c in n if f"evoked_{c}" in d) / sum(n.values())

real = np.array([cl(all_beats(D[s])) for s in subjects])
sur = np.array([cl(D[s]["surrogate"]) for s in subjects])

def band(ax, x, label, col):
    m, se = x.mean(0), x.std(0, ddof=1) / np.sqrt(len(x))
    ax.fill_between(times * 1000, m - 1.96 * se, m + 1.96 * se, color=col, alpha=.2)
    ax.plot(times * 1000, m, color=col, label=label)

fig, ax = plt.subplots(figsize=(10, 3.5))
band(ax, real, "vrais battements", "C2"); band(ax, sur, "faux battements", "grey")
ax.axhline(0, color="k", lw=.5); ax.axvline(0, color="k", lw=.5, ls="--")
ax.set_xlabel("ms après le pic R"); ax.set_ylabel("µV"); ax.legend()
ax.set_title(f"HEP fronto-central, moyenne de {len(subjects)} participants (bande = IC 95 % entre participants)")
plt.tight_layout(); plt.show()

d = mean_amp(real, times) - mean_amp(sur, times)
t, p = stats.ttest_1samp(d, 0)
print(f"Vrais − faux battements, {HEP_WIN[0]*1000:.0f}–{HEP_WIN[1]*1000:.0f} ms : {d.mean():+.2f} µV, t({len(d)-1}) = {t:.2f}, p = {p:.3f}")

In [ ]:
info = mne.create_info(ch_names, 1 / (times[1] - times[0]), "eeg")
info.set_montage(_montage(), on_missing="ignore")
ga = mne.EvokedArray(np.mean([all_beats(D[s]) for s in subjects], 0) * 1e-6, info, tmin=times[0])
ga.plot_joint(times=[0.2, 0.25, 0.3, 0.4, 0.5], title="HEP moyen du groupe (toutes vidéos)");

## 3. Haute vs basse : la courbe

Moyenne du groupe par condition, et différence haute − basse avec son intervalle de confiance
entre participants.

In [ ]:
hi_w = np.array([cl(D[s]["evoked_haute"]) for s in subjects])
lo_w = np.array([cl(D[s]["evoked_basse"]) for s in subjects])

fig, axs = plt.subplots(1, 2, figsize=(15, 3.8), gridspec_kw=dict(width_ratios=[2, 1]))
band(axs[0], lo_w, "basse", "C0"); band(axs[0], hi_w, "haute", "C3")
dm, dse = (hi_w - lo_w).mean(0), (hi_w - lo_w).std(0, ddof=1) / np.sqrt(len(subjects))
axs[0].plot(times * 1000, dm, "k--", lw=.9, label="haute − basse")
axs[0].fill_between(times * 1000, dm - 1.96 * dse, dm + 1.96 * dse, color="k", alpha=.08)
axs[0].axvspan(HEP_WIN[0] * 1000, HEP_WIN[1] * 1000, color="orange", alpha=.08)
axs[0].axhline(0, color="k", lw=.5); axs[0].legend(); axs[0].set_xlabel("ms après le pic R"); axs[0].set_ylabel("µV")
axs[0].set_title("HEP fronto-central par condition")

diff_ev = mne.EvokedArray(np.mean([D[s]["evoked_haute"] - D[s]["evoked_basse"] for s in subjects], 0) * 1e-6,
                          info, tmin=times[0])
diff_ev.plot_topomap(times=[np.mean(HEP_WIN)], average=HEP_WIN[1] - HEP_WIN[0], axes=axs[1], colorbar=False, show=False)
axs[1].set_title(f"haute − basse, {HEP_WIN[0]*1000:.0f}–{HEP_WIN[1]*1000:.0f} ms")
plt.tight_layout(); plt.show()

## 4. Amplitude

Une valeur par participant : amplitude moyenne haute − basse sur 250–450 ms (groupe fronto-central).
Test t apparié et, en complément, test de Wilcoxon (sans hypothèse de normalité).

In [ ]:
amp_d = mean_amp(hi_w, times) - mean_amp(lo_w, times)
t, p = stats.ttest_1samp(amp_d, 0); w = stats.wilcoxon(amp_d)
print(f"Amplitude haute − basse : {amp_d.mean():+.3f} µV (ET {amp_d.std(ddof=1)/np.sqrt(len(amp_d)):.3f}), "
      f"t({len(amp_d)-1}) = {t:.2f}, p = {p:.3f} ; Wilcoxon p = {w.pvalue:.3f}")
print(f"{(amp_d > 0).sum()} participants sur {len(amp_d)} ont une amplitude plus grande en haute")

fig, ax = plt.subplots(figsize=(8, 2.8))
o = np.argsort(amp_d)
ax.bar(np.array(subjects)[o], amp_d[o], color=np.where(amp_d[o] > 0, "C3", "C0"))
ax.axhline(0, color="k", lw=.5); ax.set_ylabel("haute − basse (µV)"); ax.tick_params(axis="x", rotation=90, labelsize=7)
ax.set_title("Différence d'amplitude par participant"); plt.tight_layout(); plt.show()

## 5. Latence (jackknife sur les participants)

La latence d'un HEP individuel est très bruitée (sur P001 seul : erreur-type ≈ 290 ms). La méthode
adaptée est le **jackknife** (Miller, Patterson & Ulrich, 1998 ; Ulrich & Miller, 2001) : on mesure
la latence sur la moyenne de tous les participants sauf un, on recommence en retirant chacun à son
tour, et l'on corrige la statistique t pour ce procédé.

Latence = aire fractionnelle 50 % dans 200–600 ms. La polarité de la composante (positive ou
négative) est fixée une fois pour toutes à partir de la moyenne du groupe, pour que toutes les
mesures portent sur la même composante.

In [ ]:
pol = 1 if (real.mean(0)[(times >= LAT_WIN[0]) & (times <= LAT_WIN[1])]).sum() >= 0 else -1
lat = lambda w: frac_area_latency(w, times, polarity=pol)
n = len(subjects)
full = lat(hi_w.mean(0)) - lat(lo_w.mean(0))
loo = np.array([lat(np.delete(hi_w, i, 0).mean(0)) - lat(np.delete(lo_w, i, 0).mean(0)) for i in range(n)])
se_jk = np.sqrt((n - 1) / n * ((loo - loo.mean()) ** 2).sum())
t_jk = full / se_jk if se_jk > 0 else np.nan
p_jk = 2 * stats.t.sf(abs(t_jk), n - 1)
print(f"Polarité de la composante : {'positive' if pol > 0 else 'négative'}")
print(f"Latence groupe : basse {lat(lo_w.mean(0))*1000:.0f} ms, haute {lat(hi_w.mean(0))*1000:.0f} ms")
print(f"Différence haute − basse : {full*1000:+.0f} ms, ET jackknife {se_jk*1000:.0f} ms, "
      f"t_jk({n-1}) = {t_jk:.2f}, p = {p_jk:.3f}")
print(f"(latence du pic, pour info : basse {peak_latency(lo_w.mean(0), times, polarity=pol)*1000:.0f} ms, "
      f"haute {peak_latency(hi_w.mean(0), times, polarity=pol)*1000:.0f} ms)")

## 6. Sur tout le scalp : test par clusters

Pour ne pas dépendre du choix du groupe fronto-central ni de la fenêtre 250–450 ms : test de
permutation par clusters (électrodes × temps, 0–600 ms) sur les différences haute − basse des
participants, corrigé pour les comparaisons multiples.

In [ ]:
from mne.stats import permutation_cluster_1samp_test
keep = (times >= 0) & (times <= 0.6)
step = 4                                                            # 1000 Hz -> 250 Hz
X = np.array([(D[s]["evoked_haute"] - D[s]["evoked_basse"])[:, keep][:, ::step].T for s in subjects])
adj, _ = mne.channels.find_ch_adjacency(info, "eeg")
T, clusters, pv, _ = permutation_cluster_1samp_test(X, n_permutations=2000, adjacency=adj, seed=0,
                                                    out_type="mask", verbose=False)
tt = times[keep][::step]
print(f"{len(clusters)} clusters ; plus petit p = {pv.min() if len(pv) else float('nan'):.3f}")
for i in np.argsort(pv)[:3]:
    tm, chm = np.where(clusters[i])
    print(f"  p = {pv[i]:.3f} : {tt[tm.min()]*1000:.0f}–{tt[tm.max()]*1000:.0f} ms, "
          f"{len(set(chm))} électrodes ({', '.join(sorted(set(np.array(ch_names)[chm]))[:8])}…)")

## 7. Contrôles : est-ce vraiment le cerveau ?

1. **Fréquence cardiaque** : si le cœur bat plus vite en activation haute, l'onde T et le battement
   suivant arrivent plus tôt, ce qui peut imiter un changement de latence du HEP.
2. **ECG moyen** : une différence de forme de l'ECG entre conditions peut passer dans l'EEG
   (artefact du champ cardiaque) et créer une fausse différence de HEP.

In [ ]:
res = resume.set_index("subj").loc[subjects]
rr_d = (res.rr_haute_s - res.rr_basse_s).to_numpy() * 1000
t, p = stats.ttest_1samp(rr_d, 0)
print(f"RR haute − basse : {rr_d.mean():+.1f} ms, t({len(rr_d)-1}) = {t:.2f}, p = {p:.3f}")

ecg_hi = np.array([D[s]["ecg_haute"] for s in subjects]); ecg_lo = np.array([D[s]["ecg_basse"] for s in subjects])
ecg_d = mean_amp(ecg_hi, times) - mean_amp(ecg_lo, times)
t, p = stats.ttest_1samp(ecg_d, 0)
print(f"ECG moyen {HEP_WIN[0]*1000:.0f}–{HEP_WIN[1]*1000:.0f} ms, haute − basse : t({len(ecg_d)-1}) = {t:.2f}, p = {p:.3f}")
r, p_r = stats.pearsonr(rr_d, amp_d)
print(f"Lien entre différence de RR et différence d'amplitude HEP : r = {r:.2f}, p = {p_r:.3f}")

fig, axs = plt.subplots(1, 2, figsize=(12, 3.2))
axs[0].scatter(res.rr_basse_s * 1000, res.rr_haute_s * 1000, s=15)
lim = [res[["rr_basse_s", "rr_haute_s"]].min().min() * 1000, res[["rr_basse_s", "rr_haute_s"]].max().max() * 1000]
axs[0].plot(lim, lim, "k--", lw=.7); axs[0].set_xlabel("RR basse (ms)"); axs[0].set_ylabel("RR haute (ms)")
axs[0].set_title("1. Intervalle RR par participant")
z = lambda x: (x - x[:, (times >= -0.2) & (times <= -0.05)].mean(1, keepdims=True)) / np.abs(x).max(1, keepdims=True)
band(axs[1], z(ecg_lo), "basse", "C0"); band(axs[1], z(ecg_hi), "haute", "C3")
axs[1].set_title("2. ECG moyen (normalisé par participant)"); axs[1].set_xlabel("ms après R"); axs[1].legend()
plt.tight_layout(); plt.show()

## 8. Bilan

À compléter après lecture des sorties ci-dessus :
- existe-t-il un HEP dans le groupe (section 2) ?
- l'amplitude (section 4), la latence (section 5) ou un cluster (section 6) diffèrent-ils entre
  activation haute et basse ?
- les contrôles (section 7) excluent-ils une explication par le cœur lui-même ?

**Limites** : découpage haute / basse provisoire (en attendant Micha) ; 26 participants ;
une seule fenêtre et un seul groupe d'électrodes pour les tests ciblés (le test par clusters
complète).